# Vertically Checked Tilt Analysis

Main summary plots for vertically checked tilt distance.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import seacofs_tilt_tools as tilt

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths)
df_eddies = tilt.add_time_coordinates(df_eddies)

n_profiles = len(df_eddies)
n_tilt = df_eddies.TiltDis.notna().sum()
print(f"Eddy-days: {n_profiles:,}")
print(f"Tilt-checked eddy-days: {n_tilt:,} ({n_tilt / n_profiles:.1%})")
df_eddies.head()


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3.8), constrained_layout=True)

tilt.binned_tilt_panel(axs[0], df_eddies, "Age", "Eddy age (days)", ylabel="Tilt distance (km)")
tilt.binned_tilt_panel(axs[1], df_eddies, "norm_time", "Normalised lifetime")
tilt.binned_tilt_panel(axs[2], df_eddies, "Rc", "Radius (km)")
axs[0].legend(frameon=False)
plt.show()


In [ ]:
season_df = df_eddies.copy()
if "Date" in season_df.columns:
    season_df = tilt.add_season(season_df)
    order = ["DJF", "MAM", "JJA", "SON"]
    data = [season_df.loc[season_df.Season == s, "TiltDis"].dropna() for s in order]
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.boxplot(data, tick_labels=order, showfliers=False, patch_artist=True)
    ax.set_xlabel("Season")
    ax.set_ylabel("Tilt distance (km)")
    ax.grid(True, axis="y", alpha=0.25)
    plt.show()
else:
    print("No Date column available for seasonal summary.")


In [ ]:
summary = (
    df_eddies
    .groupby("Cyc")
    .agg(eddy_days=("TiltDis", "size"), tilt_days=("TiltDis", "count"), median_tilt_km=("TiltDis", "median"), mean_tilt_km=("TiltDis", "mean"))
)
summary
